For a simple counting experiment, the expected background event is b and the observed event is n . The best estimator for signal event s is: s=n−b.

There are different metrics to evaluate discovery significance. 
SimplifiedZ 0: Z_0,simple = s/√b
Asymptotic Z0: Z_0,asymptotic = √2((s+b)ln(1+s/b)−s)
Bayesian Z0: p−value=∫∞nPoisson(k|b)dk.
Z0_Bayesian = Gauss1+sided(p−value)

In this exercise, we will implement each of the metric and compare consistency.



In [1]:
import numpy as np
import matplotlib.pyplot as plt
import iminuit.minimize as minimize
from tqdm import tqdm
import scipy
from scipy.stats import poisson, norm

# Define test statistics q_0 for Frequentist approach
# 1. We require signal event s >0 for positive signal yield.
# Therefore, the test statistics q_0 is 0 if N_obs <= Nb
# 2. Compute two Poisson loglikelihood of
# a) backgorund only model
# b) signal+background model
# Evaluate -2 log likelihood ratio between a) and b)


Implement four metrics:

Now, let's apply our code for numerical calculations. 
Consider the case that backogrund only model with yields b=0.5 and observed events n=5.
Calclate discovery significance for each of the metric, respectively.

In [2]:
#Define the four metrics Z_0 for different approaches
#Apply the test statistics q_0 is 0 if N_obs <= Nb
def SimplifiedZ0(N_obs, N_b):
    if N_b <= 0:
        return 0
    s = max(N_obs - N_b, 0)
    Zscore = s / np.sqrt(N_b)
    return float(Zscore)

def AsymptoticZ0(N_obs, N_b):
    if N_b <= 0:
        return 0
    s = max(N_obs - N_b, 0)
    if s == 0:
        return 0
    Zscore = np.sqrt(2.0 * ((s + N_b) * np.log1p(s / N_b) - s))
    return float(Zscore)

def BayesianZ0(N_obs, N_b):
    if N_b <= 0:
        return 0
    pval = poisson.sf(N_obs - 1, mu=N_b)
    z = norm.isf(pval)
    return float(max(z, 0.0))

def poisson_logl(n, mu):
    if mu <= 0:
        return 0.0 if (n == 0) else -np.inf
    return n * np.log(mu) - mu - scipy.special.gammaln(n + 1)

def q0(N_obs, N_b):
    mu_b = max(N_b, 1e-12)
    mu_sb = max(N_obs, N_b, 1e-12)
    logl_b = poisson_logl(N_obs, mu_b)
    logl_sb = poisson_logl(N_obs, mu_sb)
    q = -2.0 * (logl_b - logl_sb)
    return float(max(q, 0.0))

# Call each metric function with the given Nobs and Nb values
Nobs = 5
Nb = 0.5

simplified_result = SimplifiedZ0(Nobs, Nb)
asymptotic_result = AsymptoticZ0(Nobs, Nb)
bayesian_result = BayesianZ0(Nobs, Nb)
q0_result = q0(Nobs, Nb)
Z_q0_value = np.sqrt(q0_result)

print(f"Simplified Z_0: {simplified_result:.3f}")
print(f"Asymptotic Z_0: {asymptotic_result:.3f}")
print(f"Bayesian Z_0: {bayesian_result:.3f}")
print(f"q_0: {q0_result:.3f}")
print(f"Z from q_0: {Z_q0_value:.3f}")


Simplified Z_0: 6.364
Asymptotic Z_0: 3.745
Bayesian Z_0: 3.580
q_0: 14.026
Z from q_0: 3.745


## Describe the consistency between different metrics. ##
Write your answers here: All 4 methods aim to measure how inconsistent the observed data are with the background-only hypothesis.
When the expected background​ is large enough, all four metrics give nearly identical results.
For very small N_background we see differences! We see simplified Z_0 overestimates the value for low N_background
Asymptotic, Bayesian, and Sqrt(q_0) were more accurate and reliable.